# Exploring the datasets: SHD, SSC and DVS128 Gesture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Avi-Seth/UCLA-COSMOS-SNN-Energy-efficiency/blob/main/SHD-SSC-order-of-events/01_explore_datasets.ipynb)

**SHD** (Spiking Heidelberg Digits) and **SSC** (Spiking Speech Commands) are spoken words passed
through a model of the inner ear with 700 output channels: each channel fires spikes while its band of
the sound is active. **DVS128 Gesture** is hand and arm gestures filmed by an event camera, whose
128 x 128 pixels each fire an event when their brightness changes. `02_train_SHD.ipynb` and
`03_train_SSC.ipynb` train networks on SHD and SSC; DVS128 Gesture is explored here for comparison.

In all three datasets a recording is a list of events, each with a time and a channel. The networks
cut every recording into time windows and count the events per window and channel, which gives a table
with one row per window. This notebook looks at the raw events, at those tables (cut exactly as the
training notebooks cut them), and at what the experiments do to the tables: shuffling, reversing and
joining. The last cell puts the three datasets side by side.

**How to run it.** Runtime > Run all; no GPU is needed. The SHD and SSC files are downloaded from the
dataset authors' site (169 MB and 1.7 GB). DVS128 Gesture is off as the notebook is set; `RUN_DVS = True`
in Step 1 downloads it with the tonic library. Figures, two sound files and a summary table are shown
here and saved on the Colab machine; the last cell offers them as one `.zip` download. A part whose
files cannot be found or downloaded is skipped.

**How to read the code.** Every code cell has a short explanation above it, and the text after a `#`
on a line of code says what that line does.

**Sources.** SHD and SSC: B. Cramer, Y. Stradmann, J. Schemmel and F. Zenke, "The Heidelberg Spiking
Data Sets for the Systematic Evaluation of Spiking Neural Networks", IEEE Transactions on Neural
Networks and Learning Systems 33(7), 2744-2757 (2022), doi:10.1109/TNNLS.2020.3044364; the files are
at https://zenkelab.org/datasets/. DVS128 Gesture: A. Amir et al., "A Low Power, Fully Event-Based
Gesture Recognition System", CVPR 2017, 7243-7252. The version used here is the one distributed with
the tonic library: one .npy file per gesture, with times in milliseconds.

## Step 1. Settings

- `RUN_SHD`, `RUN_SSC` and `RUN_DVS` choose the parts to run.
- `USE_DRIVE = True` saves the figures, sounds and summary on your Google Drive instead of the Colab
  machine (which is deleted when the session ends).
- `MY_DATA_DIR` is optional: a folder that already holds the dataset files, anywhere inside it (for
  SHD and SSC the `.h5` files or their `.h5.gz` versions; for DVS128 Gesture the folders
  `ibmGestureTrain` and `ibmGestureTest`). Whatever is not found there is downloaded.

In [ ]:
RUN_SHD = True            # part 1: SHD (a 169 MB download)
RUN_SSC = True            # part 2: SSC (a 1.7 GB download; unpacking its training file takes a few minutes)
RUN_DVS = False           # part 3: DVS128 Gesture (downloaded with the tonic library)
USE_DRIVE = False         # True: save the figures, sounds and summary on your Google Drive
MY_DATA_DIR = ''          # optional: a folder with the dataset files, for example '/content/drive/MyDrive/data/'
SAMPLE = 500              # the training recording looked at in detail
DVS_EVERY = 5             # DVS128 Gesture statistics read every 5th training file (1 = all of them, slower)

## Step 2. Folders and Google Drive

Google Drive is connected only when it is needed (`USE_DRIVE`, or a `MY_DATA_DIR` on Drive); Colab then
asks once for permission.

In [ ]:
if USE_DRIVE or MY_DATA_DIR.startswith('/content/drive'):   # Google Drive is needed: connect it
    from google.colab import drive            # the tool that connects Google Drive to this notebook
    drive.mount('/content/drive')             # your Drive files appear under /content/drive/MyDrive

if USE_DRIVE:
    OUT_DIR = '/content/drive/MyDrive/shd_ssc_order/explore/'   # on your Drive: kept after the session
else:
    OUT_DIR = '/content/shd_ssc_order/explore/'                 # on this Colab machine: deleted after the session
DATA_DIR = '/content/data/'                                     # the data files, on this machine's own disk (fast)
DATA_URL = 'https://zenkelab.org/datasets/'                     # the SHD and SSC authors' site
print('figures, sounds and the summary go to', OUT_DIR)

## Step 3. Libraries and figure style

A picture of a table of counts uses white for an empty cell and blue for spikes (one spike is already
clearly blue). For the event camera, blue and orange stand for its two kinds of events.

In [ ]:
import os                                     # folders and file names
import sys                                    # the Python program running this notebook (to install tonic)
import gzip                                   # unpacking .gz files
import shutil                                 # copying files and making .zip files
import hashlib                                # checksums of the downloaded files
import subprocess                             # running pip to install tonic (part 3 only)
import urllib.request                         # downloading files
import wave                                   # writing .wav sound files
import h5py                                   # reading the .h5 files of SHD and SSC
import numpy as np                            # arrays of numbers
import matplotlib.pyplot as plt               # figures
from matplotlib.colors import LinearSegmentedColormap, ListedColormap   # colour scales
from matplotlib.ticker import MaxNLocator     # whole numbers on colour bars
from IPython.display import Audio, display    # playing a sound in the notebook
from tqdm.auto import tqdm                    # progress bars

os.makedirs(OUT_DIR, exist_ok=True)           # create the output folder if it does not exist yet
os.makedirs(DATA_DIR, exist_ok=True)          # and the data folder

INK, INK2, MUTED = '#0b0b0b', '#52514e', '#898781'        # text, secondary text, grey
BLUE, ORANGE = '#2a78d6', '#eb6834'                        # the two colours
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 9.5, 'axes.labelsize': 9,   # font sizes
                     'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK2,          # a light grey frame
                     'xtick.color': INK2, 'ytick.color': INK2,
                     'axes.spines.top': False, 'axes.spines.right': False,          # no frame on top and right
                     'figure.facecolor': 'white', 'savefig.facecolor': 'white'})    # white background
SPIKES = ListedColormap(plt.cm.Blues(np.linspace(0.3, 1, 256)), name='spikes')   # light blue to dark blue
SPIKES = SPIKES.with_extremes(under='white')  # values below the scale (empty cells) are white
POLARITY = LinearSegmentedColormap.from_list('polarity', [BLUE, 'white', ORANGE])   # -1 blue, 0 white, +1 orange
SUMMARY = {}                                  # numbers for the side-by-side table at the end


def save_figure(fig, name):
    """Save a figure in OUT_DIR as a .png file, then show it."""
    fig.savefig(OUT_DIR + name + '.png', dpi=150, bbox_inches='tight')
    plt.show()

## Step 4. Get the files

SHD has two files and SSC three (train, valid and test).

For every file the notebook looks, in this order:

1. on this machine, from earlier in the same session;
2. in `MY_DATA_DIR`, if it is set (the file or its `.gz` version, anywhere inside that folder);
3. on the dataset authors' site, https://zenkelab.org/datasets/. The compressed file is downloaded,
   checked against the checksum the site publishes (an MD5 checksum is a fingerprint of a file, so a
   damaged download is caught), and unpacked.

The files go to the Colab machine's own disk, which is fast to read. Every file is written under a
temporary name and renamed only when complete, so an interrupted copy or download is never used.

DVS128 Gesture is the version distributed with the tonic library: the folders `ibmGestureTrain` and
`ibmGestureTest` hold one `.npy` file per gesture. They are looked for inside `MY_DATA_DIR`; if they
are not there, tonic is installed and downloads them (tonic checks its own downloads). A part whose
files cannot be found or downloaded is skipped, with a message that says why.

In [ ]:
def find_in_my_data(name):
    """The path of `name` (or of `name` + '.gz') anywhere inside MY_DATA_DIR, or None if it is not there."""
    if MY_DATA_DIR == '':                                              # no folder of your own was given
        return None
    for folder, subfolders, files in os.walk(MY_DATA_DIR):            # every folder inside MY_DATA_DIR
        subfolders.sort()                                              # look through them in alphabetical order
        for candidate in [name, name + '.gz']:                         # the file itself, or its compressed version
            if candidate in files:
                return os.path.join(folder, candidate)
    return None


def download(url, path):
    """Download `url` to `path` with a progress bar; the file gets its real name only when complete."""
    with urllib.request.urlopen(url, timeout=60) as response:          # open the connection
        size = int(response.headers.get('Content-Length', 0))          # the file's size, if the site gives it
        with open(path + '.part', 'wb') as out:
            with tqdm(total=size, unit='B', unit_scale=True, desc='downloading ' + os.path.basename(path)) as bar:
                while True:
                    chunk = response.read(4 * 1024 * 1024)             # 4 MB at a time
                    if len(chunk) == 0:                                # nothing left: the download is complete
                        break
                    out.write(chunk)
                    bar.update(len(chunk))
    os.replace(path + '.part', path)                                   # the real name, now that it is complete


def md5_of(path):
    """The MD5 checksum of a file: a fingerprint that changes if a single byte of the file changes."""
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(16 * 1024 * 1024)                           # 16 MB at a time
            if len(chunk) == 0:
                break
            digest.update(chunk)
    return digest.hexdigest()


def published_md5(dataset):
    """The checksums the dataset authors publish for 'shd' or 'ssc', as {file name: checksum}."""
    with urllib.request.urlopen(DATA_URL + dataset + '/md5sums.txt', timeout=60) as response:
        text = response.read().decode()
    sums = {}
    for line in text.splitlines():                                     # every line is 'checksum  file name'
        parts = line.split()
        if len(parts) == 2:
            sums[parts[1]] = parts[0]
    return sums


def unpack(packed, path):
    """Unpack a .gz file to `path`; the file gets its real name only when complete."""
    with gzip.open(packed, 'rb') as source:
        with open(path + '.part', 'wb') as out:
            shutil.copyfileobj(source, out, 16 * 1024 * 1024)          # 16 MB at a time
    os.replace(path + '.part', path)


def get_data_file(name):
    """Put the data file `name` (for example 'shd_train.h5') on this machine, and return where it is."""
    path = DATA_DIR + name
    if os.path.exists(path):                                           # 1. here from earlier in this session
        return path
    mine = find_in_my_data(name)
    if mine is not None:                                               # 2. in MY_DATA_DIR
        print('taking', name, 'from', mine)
        if mine.endswith('.gz'):
            unpack(mine, path)                                         # a compressed file: unpack it here
        else:
            shutil.copy(mine, path + '.part')                          # copy it here, under a temporary name
            os.replace(path + '.part', path)
        return path
    dataset = name.split('_')[0]                                       # 3. from the dataset authors' site
    packed = path + '.gz'                                              # the site has compressed files only
    url = DATA_URL + dataset + '/' + name + '.gz'
    try:
        download(url, packed)
    except Exception as error:                                         # no connection, a moved file, ...
        raise RuntimeError(f'{name}.gz could not be downloaded from {url} ({error}). Download the '
                           f'{dataset.upper()} files from https://zenkelab.org/datasets/ yourself, put them in a '
                           f'folder on your Google Drive, set MY_DATA_DIR in Step 1 to that folder, and run '
                           f'the notebook again.') from None
    try:
        expected = published_md5(dataset).get(name + '.gz')            # the checksum the site publishes
    except Exception:
        expected = None                                                # the list of checksums could not be read
    if expected is None:
        print('note: no published checksum was found for', name + '.gz', '- the download could not be checked')
    elif md5_of(packed) != expected:                                   # a damaged download
        os.remove(packed)
        raise RuntimeError(f'{name}.gz was damaged while downloading (its checksum is wrong). '
                           f'Run this cell again.')
    else:
        print(name + '.gz: its checksum matches the one the dataset authors publish')
    print('unpacking', name + '.gz')
    unpack(packed, path)
    os.remove(packed)                                                  # the compressed file is no longer needed
    return path


def find_dvs_folders(top):
    """The folders ibmGestureTrain and ibmGestureTest anywhere inside `top`, as {'train': ..., 'test': ...}."""
    found = {}
    for folder, subfolders, files in os.walk(top):                # every folder inside `top`
        subfolders.sort()                                         # look through them in alphabetical order
        name = os.path.basename(folder)
        has_users = False                                         # does it hold the user folders?
        for sub in subfolders:
            if sub.startswith('user'):
                has_users = True
        if name == 'ibmGestureTrain' and has_users and 'train' not in found:
            found['train'] = folder + '/'
        if name == 'ibmGestureTest' and has_users and 'test' not in found:
            found['test'] = folder + '/'
    return found


FOUND = {'shd': False, 'ssc': False, 'dvs': False}   # which parts have their files

# SHD
if RUN_SHD:
    try:
        SHD_TRAIN = get_data_file('shd_train.h5')
        SHD_TEST = get_data_file('shd_test.h5')
        FOUND['shd'] = True
    except RuntimeError as error:
        print('SHD:', error, '- part 1 is skipped')

# SSC
SSC_VALID = None                              # the valid split is optional here
if RUN_SSC:
    try:
        SSC_TRAIN = get_data_file('ssc_train.h5')
        SSC_TEST = get_data_file('ssc_test.h5')
        FOUND['ssc'] = True
    except RuntimeError as error:
        print('SSC:', error, '- part 2 is skipped')
    if FOUND['ssc']:
        try:
            SSC_VALID = get_data_file('ssc_valid.h5')
        except RuntimeError as error:
            print('SSC valid split:', error, '- it is left out')

# DVS128 Gesture
DVS_FOLDERS = {}                              # 'train' and 'test' -> their folder
DVS_DIR = ''                                  # the folder that holds them
if RUN_DVS:
    if MY_DATA_DIR != '':                     # first in your own folder
        DVS_FOLDERS = find_dvs_folders(MY_DATA_DIR)
        DVS_DIR = MY_DATA_DIR
    if len(DVS_FOLDERS) < 2:                  # not there: download with tonic
        try:
            print('installing tonic and downloading DVS128 Gesture (this can take several minutes)')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tonic'], check=True)
            import tonic                      # the event-data library that distributes this version
            for train in [True, False]:       # the training and the test gestures
                tonic.datasets.DVSGesture(save_to=DATA_DIR + 'dvsgesture/', train=train)
            DVS_DIR = DATA_DIR + 'dvsgesture/'
            DVS_FOLDERS = find_dvs_folders(DVS_DIR)
        except Exception as error:            # no connection, a failed install, ...
            print('DVS128 Gesture could not be downloaded:', error)
    if len(DVS_FOLDERS) == 2:
        FOUND['dvs'] = True
    else:
        print('DVS128 Gesture: ibmGestureTrain and ibmGestureTest not found - part 3 is skipped')

print('parts that will run:', [p for p in FOUND if FOUND[p]])

## Part 1. SHD, Spiking Heidelberg Digits

The digits zero to nine spoken in English and in German: 20 classes. Every recording passed through a
model of the inner ear whose 700 channels each fire spikes for one band of the sound.

### 1.1 Open the two files and list what is inside

In [ ]:
if FOUND['shd']:
    shd_train = h5py.File(SHD_TRAIN, 'r')                   # open the files for reading
    shd_test = h5py.File(SHD_TEST, 'r')
    shd_train.visit(print)                                  # print every group and dataset in the file

### 1.2 Recordings, classes, and how the spikes are stored

`spikes/times` holds every recording's spike times and `spikes/units` their channels; `labels` holds
every recording's class, and `extra/keys` the names of the classes.

In [ ]:
if FOUND['shd']:
    times = shd_train['spikes']['times']                    # spike times of every training recording
    units = shd_train['spikes']['units']                    # their channels (0 to 699)
    labels = np.array(shd_train['labels']).astype(np.int64)   # every recording's class (0 to 19)
    names = []                                              # the names of the 20 classes
    for k in shd_train['extra']['keys'][:]:
        names.append(k.decode())                            # stored as bytes: decode to text
    print('recordings: train', len(labels), '| test', len(shd_test['labels']))
    print('classes:', len(names), names)
    print('training recordings per class:', np.bincount(labels, minlength=len(names)).tolist())
    print('spike times stored as', times[0].dtype, '(half precision) | channels stored as', units[0].dtype)

### 1.3 One recording in detail

A recording is two lists of the same length: the k-th time belongs to the k-th channel.

In [ ]:
def describe(t, ch, title, n_show=8, C=700):
    """Print one recording: its first spikes, and a few numbers about its channels."""
    t = np.asarray(t, dtype=np.float64)                     # spike times, in double precision
    ch = np.asarray(ch, dtype=np.int64)                     # spike channels
    counts = np.bincount(ch, minlength=C)                   # spikes per channel
    print(f'{title}: {len(t):,} spikes from {t.min():.4f} s to {t.max():.4f} s, on {np.count_nonzero(counts)} of {C} channels')
    print(f'{"spike":>7} | {"time (s)":>9} | channel')
    for k in range(min(n_show, len(t))):                    # the first few spikes
        print(f'{k:>7} | {t[k]:>9.5f} | {ch[k]:>7}')
    if len(t) > n_show:
        print(f'{"...":>7} | {len(t) - n_show:,} more')
    print(f'silent channels: {np.sum(counts == 0)} | channels with a single spike: {np.sum(counts == 1)} | '
          f'busiest channel: {counts.argmax()} with {counts.max()} spikes')
    print('times and channels are parallel lists: the k-th time belongs to the k-th channel')


if FOUND['shd']:
    i = min(SAMPLE, len(labels) - 1)                        # the recording looked at in detail
    describe(times[i], units[i], f'training recording {i}, "{names[labels[i]]}"')

### 1.4 Speakers

Test accuracy on SHD measures mostly how well a network copes with new speakers: some speakers appear
only in the test set. `extra/meta_info` holds the age, sex and height of every speaker.

In [ ]:
if FOUND['shd']:
    spk_train = np.array(shd_train['extra']['speaker']).astype(np.int64)   # the speaker of every recording
    spk_test = np.array(shd_test['extra']['speaker']).astype(np.int64)
    only_test = sorted(set(spk_test.tolist()) - set(spk_train.tolist()))  # speakers never heard in training
    test_share = float(np.isin(spk_test, only_test).mean())              # their share of the test recordings
    print('training speakers:', sorted(set(spk_train.tolist())))
    print('test speakers:', sorted(set(spk_test.tolist())))
    print('speakers only in the test set:', only_test, f'| {test_share:.0%} of the test recordings')
    if 'meta_info' in shd_train['extra']:
        info = shd_train['extra']['meta_info']
        columns = list(info.keys())                         # for example age, body_height, gender
        print('extra/meta_info holds', ', '.join(columns), '(row k is speaker k):')
        print(f'{"speaker":>7} | {"train":>5} | {"test":>5} | ' + ' | '.join(columns))
        for s in range(len(info[columns[0]])):              # one line per speaker
            values = []
            for c in columns:
                v = info[c][s]
                if isinstance(v, bytes):
                    v = v.decode()                          # text stored as bytes
                values.append(str(v).rjust(len(c)))
            print(f'{s:>7} | {np.sum(spk_train == s):>5} | {np.sum(spk_test == s):>5} | ' + ' | '.join(values))

### 1.5 One recording as a picture, and how long recordings last

A raster: one dot per spike, time to the right, channel upwards. Then the length of every training
recording, from its first spike to its last (reading all of them takes a few seconds).

In [ ]:
def first_last_count(times):
    """For every recording: the time of its first spike, of its last spike, and its number of spikes.
    h5py reads a block of recordings much faster than one recording at a time."""
    n = len(times)
    first, last = np.zeros(n), np.zeros(n)
    count = np.zeros(n, dtype=np.int64)
    for start in tqdm(range(0, n, 4096), desc='reading spike times', leave=False):
        block = times[start:start + 4096]                   # 4,096 recordings at once
        for k in range(len(block)):
            t = block[k].astype(np.float64)                 # this recording's spike times
            if len(t) > 0:
                first[start + k] = t.min()
                last[start + k] = t.max()
                count[start + k] = len(t)
    return first, last, count


def raster(ax, t, ch, title, size=0.8):
    """One dot per spike: time to the right, channel upwards."""
    ax.scatter(np.asarray(t, dtype=np.float64), ch, s=size, color=BLUE, linewidths=0, rasterized=True)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('channel')
    ax.set_title(title, loc='left')


if FOUND['shd']:
    first, last, n_spikes = first_last_count(times)
    dur = last - first                                      # every recording's length (s)
    t = np.asarray(times[i], dtype=np.float64)              # the sample recording, in double precision
    ch = np.asarray(units[i], dtype=np.int64)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[2, 1]))   # two panels
    raster(ax[0], t, ch, f'"{names[labels[i]]}" (training recording {i}): one dot per spike')
    ax[1].hist(dur, 40, color=BLUE)                         # a histogram of the lengths, 40 bars
    ax[1].set_xlabel('recording length, first to last spike (s)')
    ax[1].set_ylabel('training recordings')
    ax[1].set_title('how long recordings last', loc='left')
    save_figure(fig, 'shd_raster_and_lengths')
    print(f'length: shortest {dur.min():.2f} s, median {np.median(dur):.2f} s, longest {dur.max():.2f} s')
    print(f'first spike: at 0 s in {np.mean(first == 0):.0%} of the recordings, latest at {first.max():.3f} s')
    print(f'spikes per recording: mean {n_spikes.mean():,.0f}, fewest {n_spikes.min():,.0f}, most {n_spikes.max():,.0f}')

### 1.6 Eight words side by side: four English digits and four German ones

In [ ]:
if FOUND['shd']:
    if len(names) >= 14:
        show = [0, 1, 2, 3, 10, 11, 12, 13]                 # English zero to three, German null to drei
    else:
        show = list(range(min(8, len(names))))
    fig, ax = plt.subplots(2, 4, figsize=(15, 6), sharey=True)   # a grid of 2 x 4 panels
    for a, k in zip(ax.flat, show):
        j = int(np.flatnonzero(labels == k)[0])             # the first training recording of class k
        raster(a, times[j], units[j], names[k], size=0.4)
    fig.tight_layout()
    save_figure(fig, 'shd_eight_words')

### 1.7 Listen to the spikes

Every spike becomes a 5 ms tone whose pitch rises with its channel (100 Hz at channel 0, one octave
higher every 140 channels). This is not the recorded voice, only the spike pattern made audible.

In [ ]:
def sonify(t, ch, path, rate=16000):
    """Play the spikes as sound and save them as a .wav file."""
    t = np.asarray(t, dtype=np.float64)
    sound = np.zeros(int((t.max() + 0.05) * rate))          # silence, as long as the recording (16,000 samples a second)
    n = np.arange(80)                                       # 80 samples at 16 kHz = 5 ms
    for when, c in zip(t, ch):                              # every spike
        k = int(when * rate)                                # the sample where it starts
        piece = sound[k:k + 80]                             # 5 ms of the sound (adding to it adds to the sound)
        pitch = 100 * 2 ** (float(c) / 140)                 # its pitch in Hz
        piece += np.sin(2 * np.pi * pitch * n[:len(piece)] / rate)   # add a short tone
    sound *= 0.9 / max(np.abs(sound).max(), 1e-9)           # loudness: the loudest moment at 90%
    with wave.open(path, 'wb') as w:                        # save as a .wav file
        w.setnchannels(1)                                   # mono
        w.setsampwidth(2)                                   # 16-bit samples
        w.setframerate(rate)
        w.writeframes((sound * 32767).astype('<i2').tobytes())
    display(Audio(sound, rate=rate))                        # a play button in the notebook
    print('saved as', path)


if FOUND['shd']:
    sonify(times[i], units[i], OUT_DIR + 'shd_sample.wav')

### 1.8 Time windows, and why the times are converted to double precision first

`02_train_SHD.ipynb` cuts SHD in two ways: 25 windows spanning each recording (each lasts 1/25 of that
recording, so the width differs between recordings) and 100 windows of 14 ms (the same width for every
recording; short recordings end in empty windows). The functions below cut exactly as the training
notebooks do.

The spike times are stored in half precision (16-bit numbers), whose steps grow with time. The windows
are computed in double precision (64-bit numbers); the last lines show what would go wrong otherwise.

The pictures add up 5 neighbouring channels per row, because 700 rows do not fit on a screen (the
picture would be blurred); nothing is smoothed, and empty cells stay white.

In [ ]:
def relative_windows(t, ch, T=25, C=700):
    """Spike counts per (window, channel) for T windows spanning the recording from its first spike to
    its last: every window lasts 1/T of this recording."""
    t = np.asarray(t, dtype=np.float64)                     # double precision before any arithmetic
    ch = np.asarray(ch, dtype=np.int64)
    if len(t) > 1 and t.max() > t.min():
        window = ((t - t.min()) / (t.max() - t.min()) * T).astype(np.int64)   # each spike's window
        window = window.clip(0, T - 1)                      # the very last spike belongs to the last window
    else:
        window = np.zeros(len(t), dtype=np.int64)           # no length at all: every spike in window 0
    counts = np.bincount(window * C + ch, minlength=T * C)  # spikes in every (window, channel) cell
    return counts.reshape(T, C)                             # a table of T rows and C columns


def fixed_windows(t, ch, width_s, T=100, C=700):
    """Spike counts for T windows of width_s seconds from time 0; a spike after the last window is
    counted in the last window."""
    t = np.asarray(t, dtype=np.float64)
    ch = np.asarray(ch, dtype=np.int64)
    window = (t / width_s).astype(np.int64)                 # window number = time / width
    window = window.clip(0, T - 1)                          # spikes after the last window are counted in it
    counts = np.bincount(window * C + ch, minlength=T * C)
    return counts.reshape(T, C)


def show_windows(ax, M, title, group=5, vmax=None):
    """Draw a windows x channels table, `group` neighbouring channels added up per picture row."""
    P = M.reshape(len(M), -1, group).sum(2)                 # add up every `group` neighbouring channels
    if vmax is None:
        vmax = max(P.max(), 1)                              # the darkest blue: the largest count
    im = ax.imshow(P.T, aspect='auto', origin='lower', interpolation='nearest', cmap=SPIKES, vmin=0.5,
                   vmax=vmax, extent=(-0.5, len(M) - 0.5, 0, M.shape[1]))   # empty cells (0) are below vmin: white
    ax.set_title(title, loc='left')
    ax.set_xlabel('window')
    ax.set_ylabel(f'channel ({group} per picture row)')
    return im


if FOUND['shd']:
    X25 = relative_windows(t, ch, 25)                       # 25 windows spanning the recording
    X14 = fixed_windows(t, ch, 0.014, 100)                  # 100 windows of 14 ms
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
    im = show_windows(ax[0], X25, '25 windows spanning the recording')
    fig.colorbar(im, ax=ax[0], label='spikes (5 channels added up)', ticks=MaxNLocator(integer=True))
    im = show_windows(ax[1], X14, '100 windows of 14 ms')
    fig.colorbar(im, ax=ax[1], label='spikes (5 channels added up)', ticks=MaxNLocator(integer=True))
    save_figure(fig, 'shd_windows')
    print('every spike counted exactly once:', X25.sum() == len(t) == X14.sum())
    print(f'empty cells: {np.mean(X25 == 0):.0%} of the 25 x 700 table, {np.mean(X14 == 0):.0%} of the 100 x 700 table')
    width = 1000 * dur / 25                                 # one window's width in every recording (ms)
    print(f'one of the 25 windows lasts {width.min():.1f} to {width.max():.1f} ms depending on the recording (median {np.median(width):.1f} ms)')
    print(f'recordings with spikes after 100 x 14 ms = 1.4 s: {int(np.sum(last > 1.4))} (such spikes are counted in the last window)')

    # What half precision would do: the same 25 windows computed with the stored 16-bit times
    t16 = times[i]                                          # as stored: 16-bit
    w16 = ((t16 - t16.min()) / (t16.max() - t16.min()) * 25).astype(np.int64).clip(0, 24)
    w64 = ((t - t.min()) / (t.max() - t.min()) * 25).astype(np.int64).clip(0, 24)
    print(f'16-bit time steps: {1000 * np.spacing(np.float16(0.3)):.2f} ms near 0.3 s, {1000 * np.spacing(np.float16(0.6)):.2f} ms near 0.6 s, '
          f'{1000 * np.spacing(np.float16(1.2)):.2f} ms near 1.2 s | distinct time stamps in this recording: {len(np.unique(t16)):,} for {len(t16):,} spikes')
    print('spikes that land in another window when the arithmetic is done in 16 bits:', int((w16 != w64).sum()))

### 1.9 Shuffled, reversed and joined windows

Shuffling the windows destroys their order and keeps every channel's total; so does reversing them.
Joining three words (one, two, three by the same speaker) gives the same totals in every order. This is
why a model that only sees totals must score exactly 1/6 when asked for the order of such a chain (a
control task of `02_train_SHD.ipynb`).

In [ ]:
if FOUND['shd']:
    X_shuffled = X25[np.random.default_rng(0).permutation(25)]   # the 25 rows in a random order
    X_reversed = X25[::-1]                                  # the 25 rows from last to first
    print('channel totals unchanged:', np.array_equal(X25.sum(0), X_shuffled.sum(0)),
          np.array_equal(X25.sum(0), X_reversed.sum(0)))

    speaker = None                                          # a speaker with a recording of one, two and three
    for s in np.unique(spk_train):
        if all(np.any((labels == k) & (spk_train == s)) for k in (1, 2, 3)):
            speaker = s
            break
    if speaker is not None and len(names) > 3:
        parts = []
        for k in (1, 2, 3):                                 # this speaker's first recording of each word
            j = int(np.flatnonzero((labels == k) & (spk_train == speaker))[0])
            parts.append(relative_windows(times[j], units[j]))
        chain_123 = np.concatenate(parts)                   # one, two, three: 75 windows
        chain_321 = np.concatenate(parts[::-1])             # three, two, one
        words = ', '.join(names[k] for k in (1, 2, 3))
        print(f'joined: {words} by speaker {speaker} | {len(chain_123)} windows | '
              f'same totals in both orders: {np.array_equal(chain_123.sum(0), chain_321.sum(0))}')
        panels = [X25, X_shuffled, X_reversed, chain_123]
        titles = ['as recorded', 'windows shuffled', 'windows reversed', words + ' joined']
        top = 1                                             # one colour scale for all four pictures
        for M in panels:
            top = max(top, M.reshape(len(M), -1, 5).sum(2).max())
        fig, ax = plt.subplots(1, 4, figsize=(18, 3.6), gridspec_kw=dict(width_ratios=[1, 1, 1, 2]))
        for a, M, title in zip(ax, panels, titles):
            im = show_windows(a, M, title, vmax=top)
        for a in ax[1:]:
            a.set_ylabel('')
        fig.colorbar(im, ax=ax, label='spikes (5 channels added up)', ticks=MaxNLocator(integer=True))
        save_figure(fig, 'shd_shuffle_reverse_join')

### 1.10 A single leaky neuron driven by the recording

A spiking neuron keeps a share beta of its voltage from one window to the next, so it remembers recent
input for about -width / ln(beta). With windows spanning each recording, the same beta remembers a
different stretch of time in every recording; with fixed windows it does not. The spiking networks of
`02_train_SHD.ipynb` use beta = 0.9 per window at 25 windows, and beta = 0.9 ** (width / 28 ms) for
fixed windows of 14 ms (and of 10 ms on SSC).

The neuron below works like the neurons of those networks: in every window
`voltage = beta * voltage + input - (spike in the last window) * 1`, and it spikes when its voltage is
above 1.

In [ ]:
def leaky_neuron(inputs, beta, threshold=1.0):
    """One leaky integrate-and-fire neuron: its voltage and its spikes, window by window."""
    mem, spike = 0.0, 0.0                                   # voltage and last spike, both 0 at the start
    volts, spikes = [], []
    for x in inputs:                                        # one window at a time
        mem = beta * mem + x - spike * threshold            # leak, add the input, subtract 1 after a spike
        spike = float(mem > threshold)                      # spike when the voltage is above the threshold
        volts.append(mem)
        spikes.append(spike)
    return np.array(volts), np.array(spikes)


if FOUND['shd']:
    w25 = 1000 * dur[i] / 25                                # this recording's window width (ms)
    cases = [(X25, w25, 0.9, '25 windows'), (X14, 14.0, 0.9 ** (14 / 28), '100 windows of 14 ms')]
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.4), sharey=True)
    for a, (M, w_ms, beta, title) in zip(ax, cases):
        inputs = M.sum(1) / max(M.sum(1).max(), 1)          # spikes per window, scaled to 0..1
        volts, out = leaky_neuron(inputs, beta)
        x = (np.arange(len(inputs)) + 0.5) * w_ms / 1000    # the middle of every window, in seconds
        a.plot(x, inputs, color=MUTED, lw=1.2, label='input (spikes per window, scaled)')
        a.plot(x, volts, color=BLUE, lw=1.4, label='membrane voltage')
        a.scatter(x[out > 0], np.full(int(out.sum()), 1.0), color=ORANGE, marker='|', s=120, zorder=3, label='output spike')
        a.axhline(1.0, color=INK2, lw=0.6, ls='--')         # the threshold
        a.set_title(f'{title}: beta {beta:.3f} per window, memory about {-w_ms / np.log(beta):.0f} ms', loc='left')
        a.set_xlabel('time (s)')
    ax[1].legend(loc='upper right', frameon=False)          # the right panel has room after the recording ends
    save_figure(fig, 'shd_leaky_neuron')
    memory = width / -np.log(0.9)                           # memory at 25 windows, in every recording (ms)
    print(f'memory at 25 windows: {memory.min():.0f} to {memory.max():.0f} ms across the recordings (median {np.median(memory):.0f} ms); '
          f'at 14 ms windows: {-14 / np.log(0.9 ** (14 / 28)):.0f} ms for every recording')

    SUMMARY['SHD'] = {'recordings': f'{len(labels):,} train / {len(shd_test["labels"]):,} test',
                      'classes': len(names), 'input channels': '700',
                      'median length': f'{np.median(dur):.2f} s',
                      'events per recording (mean)': f'{n_spikes.mean():,.0f}',
                      'time stamps': f'{times[0].dtype} seconds',
                      'one of 25 windows spanning a recording': f'{np.median(width):.0f} ms (median)',
                      'new in the test set': f'{len(only_test)} speakers ({test_share:.0%} of test recordings)'}

**What to remember about SHD** (on the full files: 8,156 training and 2,264 test recordings)

- Two speakers appear only in the test set and make up about four fifths of it, so test accuracy is
  mostly accuracy on new speakers.
- Spike times are stored in half precision: convert them to double precision before any arithmetic,
  or some spikes land in a neighbouring window.
- 25 windows spanning each recording differ in width from recording to recording; fixed windows do
  not, and short recordings end in empty windows.
- Shuffling or reversing the windows, and joining words in any order, keep every channel's total. A
  model that only sees totals cannot tell the versions apart; the experiments rely on this.
- A leaky neuron's memory depends on the window width (1.10).

## Part 2. SSC, Spiking Speech Commands

Google's Speech Commands (35 English words) passed through the same inner-ear model as SHD, so the files
have the same layout. There are three splits: train, valid and test. `03_train_SSC.ipynb` does not use
the valid split: it holds out 5% of the training recordings instead.

### 2.1 Open the files and list what is inside

In [ ]:
if FOUND['ssc']:
    ssc = {'train': h5py.File(SSC_TRAIN, 'r')}              # the three splits, opened for reading
    if SSC_VALID is not None:
        ssc['valid'] = h5py.File(SSC_VALID, 'r')
    ssc['test'] = h5py.File(SSC_TEST, 'r')
    ssc['train'].visit(print)                               # every group and dataset in the file

### 2.2 Recordings, classes, and how the spikes are stored

In [ ]:
if FOUND['ssc']:
    times_ssc = ssc['train']['spikes']['times']             # spike times of every training recording
    units_ssc = ssc['train']['spikes']['units']             # their channels
    labels_ssc = np.array(ssc['train']['labels']).astype(np.int64)   # every recording's class (0 to 34)
    names_ssc = []                                          # the names of the 35 words
    for k in ssc['train']['extra']['keys'][:]:
        names_ssc.append(k.decode())
    per_class = np.bincount(labels_ssc, minlength=len(names_ssc))   # training recordings per word
    for split in ssc:
        print(f'{split:>5}: {len(ssc[split]["labels"]):,} recordings')
    print('classes:', len(names_ssc), names_ssc)
    print('training recordings per class, most to fewest:')
    order = np.argsort(-per_class, kind='stable')           # the classes from most to fewest recordings
    print(', '.join(f'{names_ssc[k]} {per_class[k]}' for k in order))
    has_info = 'meta_info' in ssc['train']['extra']
    print('spike times stored as', times_ssc[0].dtype, '| channels stored as', units_ssc[0].dtype,
          '| speaker details (extra/meta_info):', 'present' if has_info else 'none')

### 2.3 One recording in detail

In [ ]:
if FOUND['ssc']:
    i_ssc = min(SAMPLE, len(labels_ssc) - 1)
    describe(times_ssc[i_ssc], units_ssc[i_ssc], f'training recording {i_ssc}, "{names_ssc[labels_ssc[i_ssc]]}"')

### 2.4 Speakers: hashed names, and no speaker in two splits

In [ ]:
if FOUND['ssc']:
    speakers_ssc = {}                                       # split -> the speaker of every recording
    for split in ssc:
        speakers_ssc[split] = np.array(ssc[split]['extra']['speaker'])
    print('speaker names look like', speakers_ssc['train'][:3].tolist())
    for split in speakers_ssc:
        print(f'{split:>5}: {len(np.unique(speakers_ssc[split])):,} speakers')
    splits = list(speakers_ssc)
    shared = {}                                             # (split a, split b) -> speakers in both
    for a in range(len(splits)):
        for b in range(a + 1, len(splits)):
            both = set(speakers_ssc[splits[a]].tolist()) & set(speakers_ssc[splits[b]].tolist())
            shared[(splits[a], splits[b])] = len(both)
            print(f'speakers in both {splits[a]} and {splits[b]}: {len(both)}')
    per_speaker = np.unique(speakers_ssc['train'], return_counts=True)[1]   # recordings of every speaker
    print(f'recordings per training speaker: fewest {per_speaker.min()}, median {int(np.median(per_speaker))}, most {per_speaker.max()}')

### 2.5 One recording as a picture, and how long recordings last

In [ ]:
if FOUND['ssc']:
    first_ssc, last_ssc, n_spikes_ssc = first_last_count(times_ssc)
    dur_ssc = last_ssc - first_ssc                          # every recording's length (s)
    t_ssc = np.asarray(times_ssc[i_ssc], dtype=np.float64)  # the sample recording
    ch_ssc = np.asarray(units_ssc[i_ssc], dtype=np.int64)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw=dict(width_ratios=[2, 1]))
    raster(ax[0], t_ssc, ch_ssc, f'"{names_ssc[labels_ssc[i_ssc]]}" (training recording {i_ssc}): one dot per spike')
    ax[1].hist(dur_ssc, 40, color=BLUE)
    ax[1].set_yscale('log')                                 # most recordings are about 1 s: a log scale shows the rest
    ax[1].set_xlabel('recording length, first to last spike (s)')
    ax[1].set_ylabel('training recordings (log scale)')
    ax[1].set_title('how long recordings last', loc='left')
    save_figure(fig, 'ssc_raster_and_lengths')
    print(f'length: shortest {dur_ssc.min():.2f} s, median {np.median(dur_ssc):.3f} s, longest {dur_ssc.max():.2f} s | '
          f'shorter than 0.9 s: {int(np.sum(dur_ssc < 0.9)):,} of {len(dur_ssc):,}')
    print(f'first spike: at 0 s in {np.mean(first_ssc == 0):.0%} of the recordings | last spike after 1.0 s: {int(np.sum(last_ssc > 1.0))}')
    print(f'spikes per recording: mean {n_spikes_ssc.mean():,.0f}, fewest {n_spikes_ssc.min():,.0f}, most {n_spikes_ssc.max():,.0f}')

### 2.6 Eight words side by side: four common command words and four rarer words

In [ ]:
if FOUND['ssc']:
    show = []
    for w in ['yes', 'no', 'up', 'down', 'marvin', 'sheila', 'forward', 'backward']:
        if w in names_ssc:
            show.append(names_ssc.index(w))
    if len(show) == 0:
        show = list(range(min(8, len(names_ssc))))
    fig, ax = plt.subplots(2, 4, figsize=(15, 6), sharey=True)
    for a, k in zip(ax.flat, show):
        j = int(np.flatnonzero(labels_ssc == k)[0])         # the first training recording of word k
        raster(a, times_ssc[j], units_ssc[j], f'{names_ssc[k]} ({per_class[k]:,} training recordings)', size=0.4)
    fig.tight_layout()
    save_figure(fig, 'ssc_eight_words')

### 2.7 Listen to the spikes

In [ ]:
if FOUND['ssc']:
    sonify(times_ssc[i_ssc], units_ssc[i_ssc], OUT_DIR + 'ssc_sample.wav')

### 2.8 Time windows, and half precision again

`03_train_SSC.ipynb` uses 100 windows of 10 ms on SSC, which cover a recording of 1 s. Because nearly
every recording lasts about 1 s, 25 windows spanning a recording would be about 40 ms wide here. Half
precision matters even more than on SHD: its steps are about 0.5 ms just below 1 s, so many spikes
share a time stamp.

In [ ]:
if FOUND['ssc']:
    X25_ssc = relative_windows(t_ssc, ch_ssc, 25)           # 25 windows spanning the recording
    X10_ssc = fixed_windows(t_ssc, ch_ssc, 0.010, 100)      # 100 windows of 10 ms (as in the training)
    fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
    im = show_windows(ax[0], X25_ssc, '25 windows spanning the recording')
    fig.colorbar(im, ax=ax[0], label='spikes (5 channels added up)', ticks=MaxNLocator(integer=True))
    im = show_windows(ax[1], X10_ssc, '100 windows of 10 ms')
    fig.colorbar(im, ax=ax[1], label='spikes (5 channels added up)', ticks=MaxNLocator(integer=True))
    save_figure(fig, 'ssc_windows')
    print('every spike counted exactly once:', X25_ssc.sum() == len(t_ssc) == X10_ssc.sum(),
          f'| empty cells: {np.mean(X25_ssc == 0):.0%} (25 windows), {np.mean(X10_ssc == 0):.0%} (100 windows of 10 ms)')
    width_ssc = 1000 * dur_ssc / 25
    print(f'one of the 25 windows lasts {width_ssc.min():.1f} to {width_ssc.max():.1f} ms (median {np.median(width_ssc):.1f} ms)')

    # What half precision would do, for both kinds of windows
    t16 = times_ssc[i_ssc]                                  # as stored: 16-bit
    w16 = ((t16 - t16.min()) / (t16.max() - t16.min()) * 25).astype(np.int64).clip(0, 24)
    w64 = ((t_ssc - t_ssc.min()) / (t_ssc.max() - t_ssc.min()) * 25).astype(np.int64).clip(0, 24)
    f16 = (t16 / np.float16(0.010)).astype(np.int64).clip(0, 99)
    f64 = (t_ssc / 0.010).astype(np.int64).clip(0, 99)
    print(f'distinct time stamps in this recording: {len(np.unique(t16)):,} for {len(t16):,} spikes '
          f'(16-bit step just below 1 s: {1000 * np.spacing(np.float16(0.9)):.2f} ms)')
    print('spikes that land in another window when computed in 16 bits:', int((w16 != w64).sum()), '(25 windows),',
          int((f16 != f64).sum()), '(10 ms windows)')

    shared_test = shared.get(('train', 'test'), 0)          # speakers in both train and test
    recordings = f'{len(labels_ssc):,} train / {len(ssc["test"]["labels"]):,} test'
    if 'valid' in ssc:
        recordings += f' (+{len(ssc["valid"]["labels"]):,} valid)'
    if shared_test == 0:
        new_speakers = 'every speaker'
    else:
        new_speakers = f'{shared_test} speakers also in train'
    SUMMARY['SSC'] = {'recordings': recordings, 'classes': len(names_ssc), 'input channels': '700',
                      'median length': f'{np.median(dur_ssc):.2f} s',
                      'events per recording (mean)': f'{n_spikes_ssc.mean():,.0f}',
                      'time stamps': f'{times_ssc[0].dtype} seconds',
                      'one of 25 windows spanning a recording': f'{np.median(width_ssc):.0f} ms (median)',
                      'new in the test set': new_speakers}

**What to remember about SSC** (on the full files: 75,466 training, 9,981 validation and 20,382 test
recordings)

- 35 words with unbalanced classes: about 1,100 to 2,900 training recordings per word.
- No speaker is in two splits, so every test speaker is new.
- Recordings last about 1 s (median 0.994 s on the full training file), so 100 windows of 10 ms cover
  them.
- Half precision again: convert the times to double precision before cutting windows.

## Part 3. DVS128 Gesture: spikes from a camera instead of an ear

11 hand and arm gestures filmed with a 128 x 128 event camera. A pixel fires an event when its
brightness changes, with a polarity for the two directions of change (brighter or darker). Instead of
700 channels there are 128 x 128 x 2 = 32,768, and instead of a word there is a person moving. In the
version used here every gesture is its own .npy file: the folder name gives the user and the lighting,
the file name gives the class.

### 3.1 The folders

In [ ]:
def dvs_files(folder):
    """One row per gesture file: (path, class, user, lighting). The folders are called user<NN>_<lighting>
    and the files <class>.npy."""
    rows = []
    for sub in sorted(os.listdir(folder)):                  # every user folder, in alphabetical order
        if not sub.startswith('user') or not os.path.isdir(folder + sub):
            continue
        user = int(sub.split('_', 1)[0][4:])                # 'user01_led' -> 1
        light = sub.split('_', 1)[1]                        # 'user01_led' -> 'led'
        files = []
        for f in os.listdir(folder + sub):
            if f.endswith('.npy') and f[:-4].isdigit():     # '0.npy' to '10.npy'
                files.append(f)
        files.sort(key=lambda f: int(f[:-4]))               # in the order of their class numbers
        for f in files:
            rows.append((folder + sub + '/' + f, int(f[:-4]), user, light))
    return rows


if FOUND['dvs']:
    for split in ['train', 'test']:
        folder = DVS_FOLDERS[split]
        users = sorted(d for d in os.listdir(folder) if d.startswith('user'))
        print(f'{split:>5}: {folder} | {len(users)} user folders, for example {users[:3]}')
        print(f'       inside {users[0]}: {sorted(os.listdir(folder + users[0]))}')

### 3.2 One recording: what an event is

Every row of a file is one event: x (pixel column), y (pixel row), polarity (0 or 1) and time (ms).

In [ ]:
DVS_CLASSES = ['hand clapping', 'right hand wave', 'left hand wave', 'right arm clockwise',
               'right arm counter clockwise', 'left arm clockwise', 'left arm counter clockwise',
               'arm roll', 'air drums', 'air guitar', 'other gestures']    # the 11 classes, in order

if FOUND['dvs']:
    train_rows = dvs_files(DVS_FOLDERS['train'])
    test_rows = dvs_files(DVS_FOLDERS['test'])
    path, label, user, light = train_rows[min(SAMPLE, len(train_rows) - 1)]   # the sample recording
    ev = np.load(path)                                      # its events: one row each
    print(f'{path.replace(DVS_DIR, "")} | class {label}: {DVS_CLASSES[label]} | user {user} | lighting {light}')
    print(f'{len(ev):,} events in an array of shape {ev.shape} and type {ev.dtype}; one row per event:')
    print(f'{"event":>6} | {"x":>4} | {"y":>4} | {"polarity":>8} | time (ms)')
    for k in range(min(8, len(ev))):
        print(f'{k:>6} | {ev[k, 0]:>4.0f} | {ev[k, 1]:>4.0f} | {ev[k, 2]:>8.0f} | {ev[k, 3]:.3f}')
    columns = [('x', 'pixel column'), ('y', 'pixel row'), ('polarity', 'brightness went up or down'),
               ('time (ms)', 'when, in milliseconds')]
    for k in range(4):
        col, meaning = columns[k]
        print(f'{col:>9}: {ev[:, k].min():g} to {ev[:, k].max():g}, {len(np.unique(ev[:, k])):,} different values ({meaning})')
    steps = np.diff(np.unique(ev[:, 3]))                    # gaps between the distinct time stamps
    print(f'sorted in time: {bool(np.all(np.diff(ev[:, 3]) >= 0))} | smallest step between time stamps: {steps.min():g} ms | '
          f'events per ms: {len(ev) / (np.ptp(ev[:, 3]) + 1):.1f}')

    first_30ms = ev[ev[:, 3] - ev[:, 3].min() < 30]         # the events of the first 30 ms
    fig, ax = plt.subplots(figsize=(5, 4.8))
    for pol, color in ((0, BLUE), (1, ORANGE)):
        m = first_30ms[:, 2] == pol                         # the events of this polarity
        ax.scatter(first_30ms[m, 0], first_30ms[m, 1], s=3, color=color, linewidths=0, label=f'polarity {pol}')
    ax.set_xlim(-1, 128)
    ax.set_ylim(128, -1)                                    # y grows downwards, as in an image
    ax.set_aspect('equal')
    ax.set_xlabel('x (pixel column)')
    ax.set_ylabel('y (pixel row)')
    ax.set_title(f'first 30 ms on the sensor: {len(first_30ms):,} events', loc='left')
    ax.legend(loc='lower right', markerscale=3, frameon=False)
    save_figure(fig, 'dvs_first_30ms')

### 3.3 Classes, users and lighting

In [ ]:
if FOUND['dvs']:
    lab_tr = np.array([r[1] for r in train_rows])           # the class of every training recording
    lab_te = np.array([r[1] for r in test_rows])
    usr_tr = np.array([r[2] for r in train_rows])           # the user of every recording
    usr_te = np.array([r[2] for r in test_rows])
    light_tr = np.array([r[3] for r in train_rows])         # the lighting of every training recording
    print('recordings: train', len(train_rows), '| test', len(test_rows))
    print('recordings per class, train:', np.bincount(lab_tr, minlength=11).tolist(), '| test:', np.bincount(lab_te, minlength=11).tolist())
    for k in range(11):
        print(f'   class {k}: {DVS_CLASSES[k]}')
    print('training users:', sorted(set(usr_tr.tolist())))
    print('test users:', sorted(set(usr_te.tolist())), '| also in training:', sorted(set(usr_tr.tolist()) & set(usr_te.tolist())))
    for light in sorted(set(light_tr.tolist())):
        print(f'lighting {light}: {int(np.sum(light_tr == light))} training recordings')

### 3.4 A recording over time, and how long recordings last

A raster like the speech ones (time against pixel column, one dot per event; only every 20th event, or
the picture is solid ink), then the same recording cut into eight equal time slices drawn as images
(polarity 1 minus polarity 0). The statistics read every `DVS_EVERY`-th training file.

In [ ]:
def dvs_windows(ev, T=25, pool=4, size=128):
    """Event counts per (window, polarity, y, x) for T windows spanning the recording, with pool x pool
    pixels added together. T = 25, pool = 4 gives 25 windows of 2 x 32 x 32 = 2,048 channels."""
    t = ev[:, 3].astype(np.float64)                         # event times (ms)
    span = t.max() - t.min()                                # the recording's length
    if span > 0:
        window = ((t - t.min()) / span * T).astype(np.int64).clip(0, T - 1)   # each event's window
    else:
        window = np.zeros(len(t), dtype=np.int64)
    n = size // pool                                        # pixels per side after pooling (32)
    p = (ev[:, 2] > 0).astype(np.int64)                     # polarity 0 or 1
    y = ev[:, 1].astype(np.int64) // pool                   # pooled pixel row
    x = ev[:, 0].astype(np.int64) // pool                   # pooled pixel column
    cell = ((window * 2 + p) * n + y) * n + x               # one number per (window, polarity, y, x)
    counts = np.bincount(cell, minlength=T * 2 * n * n)
    return counts.reshape(T, 2, n, n)


if FOUND['dvs']:
    sub = ev[::20]                                          # every 20th event
    fig, ax = plt.subplots(figsize=(10, 3.2))
    for pol, color in ((0, BLUE), (1, ORANGE)):
        m = sub[:, 2] == pol
        ax.scatter(sub[m, 3] / 1000, sub[m, 0], s=0.6, color=color, linewidths=0, rasterized=True, label=f'polarity {pol}')
    ax.set_xlabel('time (s)')
    ax.set_ylabel('x (pixel column)')
    ax.set_title(f'{DVS_CLASSES[label]}: every 20th event', loc='left')
    ax.legend(loc='upper right', markerscale=6, frameon=False)
    save_figure(fig, 'dvs_raster')

    slices = dvs_windows(ev, T=8, pool=1)                   # 8 time slices, every pixel
    diff = slices[:, 1] - slices[:, 0]                      # polarity 1 minus polarity 0
    v = max(np.percentile(np.abs(diff), 99.5), 1)           # the colour scale (ignoring the rarest extremes)
    fig, ax = plt.subplots(1, 8, figsize=(17, 2.5))
    for k in range(8):
        ax[k].imshow(diff[k], cmap=POLARITY, vmin=-v, vmax=v, interpolation='nearest')
        ax[k].set_title(f'slice {k + 1}')
        ax[k].axis('off')
    fig.suptitle(f'{DVS_CLASSES[label]}: eight equal time slices, polarity 1 (orange) minus polarity 0 (blue)')
    save_figure(fig, 'dvs_time_slices')

    sampled = train_rows[::max(DVS_EVERY, 1)]               # every DVS_EVERY-th training file
    dur_dvs = np.zeros(len(sampled))                        # length of every file read (s)
    n_events = np.zeros(len(sampled), dtype=np.int64)       # events of every file read
    class_totals = np.zeros((11, 2 * 32 * 32))              # where the events of every class fall (for 3.8)
    class_n = np.zeros(11)                                  # files read per class
    for k in tqdm(range(len(sampled)), desc='reading DVS files', leave=False):
        p, lab = sampled[k][0], sampled[k][1]
        e = np.load(p)
        dur_dvs[k] = np.ptp(e[:, 3]) / 1000                 # first to last event, in seconds
        n_events[k] = len(e)
        class_totals[lab] += dvs_windows(e, T=1).ravel() / len(e)   # where this recording's events fall, as shares
        class_n[lab] += 1
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.hist(dur_dvs, 30, color=BLUE)
    ax.set_xlabel('recording length (s)')
    ax.set_ylabel('training recordings read')
    ax.set_title('how long recordings last', loc='left')
    save_figure(fig, 'dvs_lengths')
    print(f'read {len(sampled)} of {len(train_rows)} training recordings')
    print(f'length: shortest {dur_dvs.min():.2f} s, median {np.median(dur_dvs):.2f} s, longest {dur_dvs.max():.2f} s')
    print(f'events per recording: mean {n_events.mean():,.0f}, fewest {n_events.min():,}, most {n_events.max():,} | '
          f'mean events per second: {np.mean(n_events / dur_dvs):,.0f}')

### 3.5 All eleven gestures, from one user under one lighting

In [ ]:
if FOUND['dvs']:
    by_folder = {}                                          # (user, lighting) -> its files
    for r in train_rows:
        key = (r[2], r[3])
        if key not in by_folder:
            by_folder[key] = []
        by_folder[key].append(r)
    best_key = None                                         # the folder with the most files
    for key in by_folder:
        if best_key is None or len(by_folder[key]) > len(by_folder[best_key]):
            best_key = key
    f_user, f_light = best_key
    f_rows = sorted(by_folder[best_key], key=lambda r: r[1])   # its files, by class
    fig, ax = plt.subplots(2, 6, figsize=(17, 6))
    for a in ax.flat:
        a.axis('off')                                       # pictures, no axes
    for a, r in zip(ax.flat, f_rows):
        whole = dvs_windows(np.load(r[0]), T=1, pool=1)[0]  # the whole recording as one window
        d = whole[1] - whole[0]                             # polarity 1 minus polarity 0
        v = max(np.percentile(np.abs(d), 99.5), 1)
        a.imshow(d, cmap=POLARITY, vmin=-v, vmax=v, interpolation='nearest')
        a.set_title(f'{r[1]}: {DVS_CLASSES[r[1]]}', fontsize=8.5)
    fig.suptitle(f'user {f_user}, {f_light}: polarity 1 minus polarity 0 over the whole recording')
    fig.tight_layout()
    save_figure(fig, 'dvs_eleven_gestures')

### 3.6 Time windows for an event camera

The same 25 windows spanning each recording as for speech, but every window is now a small image with
two polarities. Adding up 4 x 4 pixels gives 2 x 32 x 32 = 2,048 channels per window. Recordings last
seconds, so these windows are several times wider than for speech.

In [ ]:
if FOUND['dvs']:
    X_dvs = dvs_windows(ev)                                 # 25 windows x 2 polarities x 32 x 32
    flat = X_dvs.reshape(len(X_dvs), -1)                    # 25 windows x 2,048 channels
    print(f'25 windows: shape {X_dvs.shape} = {flat.shape[1]:,} channels per window | every event counted once: {flat.sum() == len(ev)} | '
          f'empty cells: {np.mean(flat == 0):.0%}')
    width_dvs = 1000 * dur_dvs / 25                         # one window's width in every file read (ms)
    print(f'one of the 25 windows lasts {width_dvs.min():.0f} to {width_dvs.max():.0f} ms across the recordings read (median {np.median(width_dvs):.0f} ms)')
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    im = show_windows(ax, flat, '25 windows x 2,048 channels', group=16)
    fig.colorbar(im, ax=ax, label='events (16 channels added up)', ticks=MaxNLocator(integer=True))
    save_figure(fig, 'dvs_windows')
    fig, ax = plt.subplots(1, 9, figsize=(17, 2.4))
    for a, k in zip(ax, range(0, 25, 3)):                   # windows 1, 4, 7, ..., 25
        d = X_dvs[k, 1] - X_dvs[k, 0]
        v = max(np.abs(d).max(), 1)
        a.imshow(d, cmap=POLARITY, vmin=-v, vmax=v, interpolation='nearest')
        a.set_title(f'window {k + 1}')
        a.axis('off')
    fig.suptitle('every third window, 32 x 32 pixels after pooling, polarity 1 minus polarity 0')
    save_figure(fig, 'dvs_window_images')

### 3.7 Shuffled, reversed and joined windows (the same operations as in 1.9)

In [ ]:
if FOUND['dvs']:
    D_shuffled = flat[np.random.default_rng(0).permutation(25)]
    D_reversed = flat[::-1]
    print('channel totals unchanged:', np.array_equal(flat.sum(0), D_shuffled.sum(0)), np.array_equal(flat.sum(0), D_reversed.sum(0)))
    three = f_rows[:3]                                      # three gestures of the user in 3.5
    parts = []
    for r in three:
        parts.append(dvs_windows(np.load(r[0])).reshape(25, -1))
    chain_abc = np.concatenate(parts)                       # joined in one order ...
    chain_cba = np.concatenate(parts[::-1])                 # ... and in the reverse order
    print(f'joined: {", ".join(DVS_CLASSES[r[1]] for r in three)} | {len(chain_abc)} windows | '
          f'same totals in both orders: {np.array_equal(chain_abc.sum(0), chain_cba.sum(0))}')
    panels = [flat, D_shuffled, D_reversed, chain_abc]
    top = 1                                                 # one colour scale for all four pictures
    for M in panels:
        top = max(top, M.reshape(len(M), -1, 16).sum(2).max())
    fig, ax = plt.subplots(1, 4, figsize=(18, 3.6), gridspec_kw=dict(width_ratios=[1, 1, 1, 2]))
    titles = ['as recorded', 'windows shuffled', 'windows reversed', 'three gestures joined']
    for a, M, title in zip(ax, panels, titles):
        im = show_windows(a, M, title, group=16, vmax=top)
    for a in ax[1:]:
        a.set_ylabel('')
    fig.colorbar(im, ax=ax, label='events (16 channels added up)', ticks=MaxNLocator(integer=True))
    save_figure(fig, 'dvs_shuffle_reverse_join')
    print('Reversing the windows keeps each polarity as it is; a video played backwards would also swap them '
          '(getting brighter becomes getting darker).')

### 3.8 How alike are the event totals of the two circling directions?

The clockwise and counter clockwise arm circles trace the same circle in opposite directions. If their
event totals per pixel are as alike as those of two different gestures, or more alike, a model that
sees only totals has little to tell them apart by, and the order of the windows carries the difference.
This compares the class averages of the training files read in 3.4 (cosine similarity of where the
events fall: 1 means the same pattern); it says nothing yet about single recordings.

In [ ]:
if FOUND['dvs']:
    seen = np.flatnonzero(class_n > 0)                      # the classes with at least one file read
    avg = class_totals[seen] / class_n[seen, None]          # every class's average pattern
    avg = avg / np.linalg.norm(avg, axis=1, keepdims=True)  # scaled to length 1
    S = avg @ avg.T                                         # cosine similarity of every pair of classes
    pos = {}                                                # class -> its row in S
    for k in range(len(seen)):
        pos[seen[k]] = k
    pairs = [(3, 4), (5, 6)]                                # right arm cw/ccw, left arm cw/ccw
    others = []                                             # the similarities of all other pairs
    for a in seen:
        for b in seen:
            if a < b and (a, b) not in pairs:
                others.append(S[pos[a], pos[b]])
    for a, b in pairs:
        if a in pos and b in pos:
            print(f'{DVS_CLASSES[a]} vs {DVS_CLASSES[b]}: {S[pos[a], pos[b]]:.3f}')
    if len(others) > 0:
        print(f'all other pairs of gestures: median {np.median(others):.3f}, from {min(others):.3f} to {max(others):.3f}')
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    im = ax.imshow(S, cmap='Blues', vmin=min(others + [S.min()]), vmax=1, interpolation='nearest')
    ax.set_xticks(range(len(seen)))
    ax.set_yticks(range(len(seen)))
    ax.set_xticklabels([DVS_CLASSES[c] for c in seen], rotation=60, ha='right', fontsize=7.5)
    ax.set_yticklabels([DVS_CLASSES[c] for c in seen], fontsize=7.5)
    for a, b in pairs:                                      # orange frames around the two circling pairs
        if a in pos and b in pos:
            for r, c in ((pos[a], pos[b]), (pos[b], pos[a])):
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec=ORANGE, lw=1.6))
    ax.set_title('similarity of the classes\' average event totals\n(orange: the two circling directions)', loc='left')
    fig.colorbar(im, ax=ax, label='cosine similarity')
    save_figure(fig, 'dvs_class_similarity')

    shared_users = set(usr_tr.tolist()) & set(usr_te.tolist())
    if len(shared_users) == 0:
        new_users = 'every user'
    else:
        new_users = f'{len(shared_users)} users also in train'
    SUMMARY['DVS128 Gesture'] = {'recordings': f'{len(train_rows):,} train / {len(test_rows):,} test',
                                 'classes': len(set(lab_tr.tolist())),
                                 'input channels': '128 x 128 x 2 = 32,768 (2,048 after 4 x 4 pooling)',
                                 'median length': f'{np.median(dur_dvs):.2f} s',
                                 'events per recording (mean)': f'{n_events.mean():,.0f}',
                                 'time stamps': f'{ev.dtype} milliseconds, steps of {steps.min():g} ms',
                                 'one of 25 windows spanning a recording': f'{np.median(width_dvs):.0f} ms (median)',
                                 'new in the test set': new_users}

**What to remember about DVS128 Gesture** (on the full files: 1,077 training and 264 test recordings)

- One file is one gesture by one user under one lighting; the test users are not training users.
- 32,768 channels against 700 for speech, so pixels are pooled before a dense network sees them.
- Recordings last seconds, so 25 windows spanning a recording are much wider than for speech.
- Shuffling, reversing and joining keep every channel's total, as for speech.

## Side by side

The numbers collected above, one column per dataset; also saved as `summary.md`.

In [ ]:
if len(SUMMARY) > 0:
    datasets = list(SUMMARY)                                # the column names
    rows = []                                               # the row names, in the order first seen
    for d in datasets:
        for k in SUMMARY[d]:
            if k not in rows:
                rows.append(k)
    lines = ['| | ' + ' | '.join(datasets) + ' |', '|' + '---|' * (len(datasets) + 1)]   # a Markdown table
    for k in rows:
        cells = []
        for d in datasets:
            cells.append(str(SUMMARY[d].get(k, '')))
        lines.append(f'| {k} | ' + ' | '.join(cells) + ' |')
    with open(OUT_DIR + 'summary.md', 'w') as f:
        f.write('\n'.join(lines) + '\n')
    w = max(len(k) for k in rows)                           # the width of the first column
    widths = []                                             # the width of every dataset's column
    for d in datasets:
        longest = len(d)
        for k in rows:
            longest = max(longest, len(str(SUMMARY[d].get(k, ''))))
        widths.append(longest)
    print(f'{"":>{w}} | ' + ' | '.join(f'{d:<{c}}' for d, c in zip(datasets, widths)))
    for k in rows:
        print(f'{k:>{w}} | ' + ' | '.join(f'{str(SUMMARY[d].get(k, "")):<{c}}' for d, c in zip(datasets, widths)))
    print('\nfigures, sounds and summary.md are in', OUT_DIR)

## Where the outputs are

On your Google Drive with `USE_DRIVE = True`. Otherwise they are on this Colab machine, which is deleted
when the session ends, so they are packed into one `.zip` file and offered as a download.

In [ ]:
if USE_DRIVE:
    print('The figures, sounds and summary are on your Google Drive, in', OUT_DIR)
else:
    archive = shutil.make_archive(OUT_DIR.rstrip('/'), 'zip', OUT_DIR)   # one .zip file of the folder
    print('The figures, sounds and summary are in', archive)
    try:
        from google.colab import files        # Colab's tool for saving a file to your computer
        files.download(archive)               # your browser saves the .zip file
    except ImportError:                       # not on Colab: the file stays where it is
        pass